# 04 — Anti-Spoofing (Liveness Detection)

Uses **MiniFASNetV2** pre-trained weights from minivision-ai.  
No training required — weights are downloaded automatically on first run.

Evaluation is done on real faces from `classification_data` to verify the model detects them as real.

In [ ]:
import os
import random
import urllib.request
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT   = Path("../data")
WEIGHTS_DIR = Path("../weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Model Definition (MiniFASNetV2)

In [ ]:
class _DW(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        return F.relu(self.bn2(self.pw(F.relu(self.bn1(self.dw(x))))))


class MiniFASNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(32)
        self.layers = nn.Sequential(
            _DW(32, 64), _DW(64, 128, 2), _DW(128, 128),
            _DW(128, 256, 2), _DW(256, 256), _DW(256, 512, 2),
            *[_DW(512, 512) for _ in range(5)],
            _DW(512, 1024, 2), _DW(1024, 1024),
        )
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        return self.classifier(self.pool(self.layers(x)).flatten(1))

## Download Pre-trained Weights & Load Model

In [ ]:
WEIGHTS_URL  = ("https://huggingface.co/minivision-ai/silent-face-anti-spoofing"
                "/resolve/main/MiniFASNetV2.pth")
weights_path = WEIGHTS_DIR / "anti_spoofing.pth"

if not weights_path.exists():
    print("Downloading MiniFASNetV2 weights...")
    urllib.request.urlretrieve(WEIGHTS_URL, str(weights_path))
    print("Done.")
else:
    print(f"Weights found: {weights_path}")

model = MiniFASNet(num_classes=2)
state = torch.load(str(weights_path), map_location=DEVICE)
if "state_dict" in state:
    state = state["state_dict"]
model.load_state_dict(state, strict=False)
model.to(DEVICE).eval()
print("Model loaded.")

## Inference & Demo on Real Faces

In [ ]:
def preprocess(img_rgb):
    img = cv2.resize(img_rgb, (80, 80)).astype(np.float32) / 255.0
    img = (img - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float().to(DEVICE)

@torch.no_grad()
def predict_liveness(img_rgb):
    probs   = torch.softmax(model(preprocess(img_rgb)), dim=1)[0]
    real_p  = float(probs[1])
    return ("real" if real_p > 0.5 else "fake", real_p)

# Sample real faces from val_data
real_dir = DATA_ROOT / "classification_data" / "val_data"
samples  = random.sample(list(real_dir.glob("**/*.jpg")), 10)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, path in zip(axes.flat, samples):
    img = np.array(Image.open(path).convert("RGB"))
    label, conf = predict_liveness(img)
    ax.imshow(img)
    ax.set_title(f"{label}\n{conf:.2%}", color="green" if label == "real" else "red", fontsize=8)
    ax.axis("off")

plt.suptitle("Anti-Spoofing on Real Faces (should all be REAL)", fontsize=12)
plt.tight_layout()
plt.show()

## Accuracy on Val Set (Real Faces = all should be Real)

In [ ]:
all_imgs = list((DATA_ROOT / "classification_data" / "val_data").glob("**/*.jpg"))
sample   = random.sample(all_imgs, min(200, len(all_imgs)))

correct = 0
for path in sample:
    img = np.array(Image.open(path).convert("RGB"))
    label, _ = predict_liveness(img)
    if label == "real":
        correct += 1

print(f"Real-face detection accuracy: {correct}/{len(sample)} = {correct/len(sample):.1%}")
print("(These are all genuine faces — model should classify as 'real')")